# NB3 — Extração de Features (Segmento Completo 40min)

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

## Fluxo

```
NB1 → data/signals/{dataset}__{pat}__ctx{N}.npz   (inter 40min + pre 40min, 256Hz)
NB2 → data/preictal_estimate.json                 (PRE_SEC por paciente — usado no NB4)
                          ↓
NB3 → para cada paciente:
        carrega todos os .npz de contexto
        mapeia canais → ordem padronizada (R5→R0)
        extrai 19 features/canal em janelas 30s/passo 15s
        salva X_pre, X_inter com timestamps t_pre
        → data/features/{dataset}__{pat}.npz
                          ↓
NB4 → para cada janela W ∈ {W1=9, W2=10, W3=13, W4=19, W5=PRE_SEC individual}:
        filtra X_pre[t_pre >= -W] e X_inter[-W:]
        LOSO por contexto de crise
```

## Por que NB3 NÃO seleciona janelas pré-ictais

As janelas W (8, 10, 13, 19 min) são a **variável independente da Etapa 1 do NB4** —
são derivadas empiricamente da distribuição PRE_SEC do NB2. Colocá-las no NB3 violaria
o princípio de controle de variáveis: uma mudança nos valores de W exigiria re-executar
o NB3 inteiro. O correto é:

- **NB3** = preprocessing puro: extrai features do segmento completo (40min),
  armazena com timestamps `t_pre` (segundos antes do onset, negativos)
- **NB4** = experimento: filtra janelas pelos últimos W segundos de `t_pre`

Isso também reduz de 5 arquivos/paciente para **1 arquivo/paciente**.

## Duas janelas distintas — não confundir

| Conceito | Valor | Onde é usado |
|----------|-------|-------------|
| Janela de **segmentação** | 30s / passo 15s | NB3 — como as features são extraídas |
| Janela **pré-ictal experimental** W | 8, 10, 13, 19 min + individual | NB4 — variável do experimento |

## Ordenação de canais (R5→R0 via fatiamento de colunas)

| Col. 0–37 | 2 ch | R0 | Temporal primário (behind-ear) |
|-----------|------|----|-------------------------------|
| Col. 0–75 | 4 ch | R1 | Temporal completo |
| Col. 0–208| 11 ch| R2 | + Frontal |
| Col. 0–246| 13 ch| R3 | + Central |
| Col. 0–284| 15 ch| R4 | + Parietal |
| Col. 0–322| 17 ch| R5 | + Occipital (completo) |

SeizeIT2 (2 ch): `level_slices = {'R0': 38}` apenas.

In [1]:
# Instala dependências (execute apenas se necessário)
import subprocess, sys
for pkg in ['tqdm', 'PyWavelets', 'scipy']:
    try:
        __import__(pkg.replace('PyWavelets','pywt'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependências OK.')

Dependências OK.


## 1. Imports e configuração

In [2]:
import os, re, json, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
from scipy.stats import skew, kurtosis as kurt
from scipy.signal import welch
from scipy.integrate import trapezoid
import pywt

warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print('Instale tqdm para barra de progresso: pip install tqdm')
    # fallback silencioso
    def tqdm(it, **kw): return it
os.environ['LOKY_MAX_CPU_COUNT'] = '1'

# ── Diretórios ────────────────────────────────────────────────────────────────
ROOT       = Path('data')
SIGNAL_DIR = ROOT / 'signals'    # .npz do NB1
FEAT_DIR   = ROOT / 'features'   # saída do NB3
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# ── Parâmetros de segmentação (FIXOS — não são as janelas do experimento) ─────
SFREQ      = 256.0   # Hz — reamostragem do NB1
WIN_SEG_S  = 30.0    # janela de segmentação de features (segundos)
STEP_SEG_S = 15.0    # passo de segmentação (segundos)
MAX_PRE_S  = 40 * 60 # duração máxima do pre-ictal no .npz (do NB1)
N_FEATS_CH = 19      # features por canal

# ── Slices de coluna por nível (NB4 usa para fatiar X) ───────────────────────
# X tem n_ch_total × 19 colunas; as primeiras k×19 = nível com k canais
LEVEL_SLICES = {
    'R0':  2 * N_FEATS_CH,   #  38 colunas
    'R1':  4 * N_FEATS_CH,   #  76
    'R2': 11 * N_FEATS_CH,   # 209
    'R3': 13 * N_FEATS_CH,   # 247
    'R4': 15 * N_FEATS_CH,   # 285
    'R5': 17 * N_FEATS_CH,   # 323
}

print('Configuração carregada.')
print(f'Janela de SEGMENTAÇÃO: {WIN_SEG_S}s / passo {STEP_SEG_S}s  (≠ janelas do experimento W)')
print(f'Segmento completo do .npz: {MAX_PRE_S//60:.0f}min pre-ictal + {MAX_PRE_S//60:.0f}min interictal')
print(f'LEVEL_SLICES: {LEVEL_SLICES}')
print()
print('NOTA: as janelas W do experimento (8, 10, 13, 19 min + individual)')
print('      são definidas NO NB4 a partir dos resultados do NB2.')

Configuração carregada.
Janela de SEGMENTAÇÃO: 30.0s / passo 15.0s  (≠ janelas do experimento W)
Segmento completo do .npz: 40min pre-ictal + 40min interictal
LEVEL_SLICES: {'R0': 38, 'R1': 76, 'R2': 209, 'R3': 247, 'R4': 285, 'R5': 323}

NOTA: as janelas W do experimento (8, 10, 13, 19 min + individual)
      são definidas NO NB4 a partir dos resultados do NB2.


## 2. Ordenação de canais — R0 ⊂ R1 ⊂ … ⊂ R5

In [3]:
# Ordem aninhada: primeiros k canais = nível Rk
# → NB4 faz X[:, :k*19] para obter o nível desejado sem recalcular

# Standard 10-20 referencial (Siena, Mendeley)
CH_STD = [
    'T7',  'T8',                              # R0 (pos 0-1): behind-ear equiv.
    'P7',  'P8',                              # R1 adicional (pos 2-3)
    'Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz',  # R2 adicional (pos 4-10)
    'C3',  'C4',                              # R3 adicional (pos 11-12)
    'P3',  'P4',                              # R4 adicional (pos 13-14)
    'O1',  'O2',                              # R5 adicional (pos 15-16)
]  # 17 canais

# CHB-MIT montagem bipolar
CH_CHBMIT = [
    'T7-P7',  'T8-P8',                        # R0 (pos 0-1)
    'P7-O1',  'P8-O2',                        # R1 adicional (pos 2-3)
    'FP1-F7', 'FP2-F8', 'FP1-F3', 'FP2-F4', 'FZ-CZ', 'F7-T7', 'F8-T8',  # R2
    'F3-C3',  'F4-C4',                        # R3 adicional (pos 11-12)
    'C3-P3',  'C4-P4',                        # R4 adicional (pos 13-14)
    'P3-O1',  'P4-O2',                        # R5 adicional (pos 15-16)
]  # 17 canais

CH_ALIASES = {
    'T3': 'T7', 'T4': 'T8', 'T5': 'P7', 'T6': 'P8',
    'FP1': 'Fp1', 'FP2': 'Fp2', 'FZ': 'Fz',
}

def normalize_ch(name):
    n = name.strip()
    for pfx in ['EEG ', 'EEG-', 'eeg_']:
        if n.upper().startswith(pfx.upper()):
            n = n[len(pfx):]
    for sfx in ['-REF', '-A1', '-A2', '-LE', '-A', '-CLE', '-LF', '-AVG']:
        if n.upper().endswith(sfx.upper()):
            n = n[:-len(sfx)]
    n = n.strip()
    n_up = n.upper()
    if n_up in CH_ALIASES:
        return CH_ALIASES[n_up]
    if '-' in n:
        return n.upper()
    return n[0].upper() + n[1:].lower() if len(n) > 1 else n.upper()

def map_channels(ch_names_avail, priority_list, max_ch=17):
    """Mapeia canais disponíveis para a lista de prioridade (ordem aninhada).
    Canais ausentes recebem índice None → preenchidos com zeros.
    """
    avail_norm = {normalize_ch(c): i for i, c in enumerate(ch_names_avail)}
    used_names, used_idx, n_miss = [], [], 0
    for orig in priority_list[:max_ch]:
        pn = normalize_ch(orig)
        if pn in avail_norm:
            used_names.append(ch_names_avail[avail_norm[pn]])
            used_idx.append(avail_norm[pn])
        else:
            used_names.append(f'[{orig}]')
            used_idx.append(None)
            n_miss += 1
    return used_names, used_idx, n_miss

def choose_priority(dataset):
    if dataset == 'CHBMIT':   return CH_CHBMIT
    if dataset == 'SeizeIT2': return None   # tratado separadamente
    return CH_STD  # Siena, Mendeley

print('Mapeamento de canais definido.')

Mapeamento de canais definido.


## 3. Extração de features — 19 por canal (idênticas ao NB2)

In [4]:
def extract_features_window(window, sfreq):
    """19 features por canal. window: (n_ch, n_amostras).
    Idêntica ao NB2 — consistência garantida no código.
    """
    feats = []
    for ch in window:
        std    = np.std(ch)
        diff1  = np.diff(ch)
        std_d1 = np.std(diff1)
        mob    = std_d1 / (std + 1e-10)
        # Temporal (7)
        feats += [std, np.var(ch), float(np.sqrt(np.mean(ch**2))),
                  float(np.sum(np.abs(diff1))), float(mob),
                  float(skew(ch)), float(kurt(ch))]
        # Espectral (7)
        f, psd = welch(ch, fs=sfreq, nperseg=min(len(ch), int(sfreq*4)))
        def bp(lo, hi):
            m = (f>=lo)&(f<=hi)
            return float(trapezoid(psd[m], f[m])) if m.any() else 0.0
        d,t,a,b,g = bp(.5,4),bp(4,8),bp(8,13),bp(13,30),bp(30,40)
        tot = d+t+a+b+g+1e-10
        pn  = psd/(psd.sum()+1e-10); pn = pn[pn>0]
        feats += [d,t,a,b,g, float(-np.sum(pn*np.log2(pn))), float(b/tot)]
        # Wavelet DWT db4 (4)
        c = pywt.wavedec(ch, 'db4', level=4)
        feats += [float(np.sum(c[4]**2)), float(np.sum(c[3]**2)),
                  float(np.sum(c[2]**2)), float(np.sum(c[1]**2))]
        # Complexidade Hjorth (1)
        diff2  = np.diff(diff1)
        mob_d1 = np.std(diff2)/(std_d1+1e-10)
        feats.append(float(mob_d1/(mob+1e-10)))
    return np.array(feats, dtype=np.float32)


def segment_signal(signal, sfreq, win_s=WIN_SEG_S, step_s=STEP_SEG_S):
    """Segmenta (n_ch, n_samp) em janelas e extrai features.
    Retorna:
      feat_matrix : (n_win, n_ch×19)
      t_centers_s : (n_win,) tempo do centro de cada janela em segundos
                    (relativo ao início do segmento, positivo e crescente)
    """
    win_n  = int(win_s  * sfreq)
    step_n = int(step_s * sfreq)
    n_samp = signal.shape[1]
    feats, centers = [], []
    for start in range(0, n_samp - win_n + 1, step_n):
        feats.append(extract_features_window(signal[:, start:start+win_n], sfreq))
        centers.append((start + win_n/2) / sfreq)
    n_ch = signal.shape[0]
    if feats:
        return np.array(feats, dtype=np.float32), np.array(centers)
    return np.empty((0, n_ch*N_FEATS_CH), dtype=np.float32), np.array([])


print(f'Features OK — {N_FEATS_CH} por canal, janela de segmentação {WIN_SEG_S}s.')

Features OK — 19 por canal, janela de segmentação 30.0s.


## 4. Processamento por paciente — segmento completo 40min

In [5]:
def process_patient(dataset, pat, npz_files, overwrite=False, progress_bar=None):
    """Extrai features do segmento COMPLETO (40min) de todos os contextos.

    Checkpoint por contexto: salva parciais em CACHE_DIR e faz merge no final.
    Se interrompido, reinicia apenas os contextos ainda não processados.

    Saída final: FEAT_DIR/{dataset}__{pat}.npz com:
      X_inter, X_inter, t_inter, t_pre, ctx_ids_pre, ctx_ids_inter, meta
    """
    out_path   = FEAT_DIR / f'{dataset}__{pat}.npz'
    cache_dir  = FEAT_DIR / '.cache' / f'{dataset}__{pat}'

    if out_path.exists() and not overwrite:
        if progress_bar:
            progress_bar.update(len(npz_files))
        return str(out_path), 'ja existe'

    cache_dir.mkdir(parents=True, exist_ok=True)

    priority   = choose_priority(dataset)
    is_seizeit = (dataset == 'SeizeIT2')

    ch_order     = None
    n_miss_total = 0

    # ── Processa cada contexto individualmente (com checkpoint) ─────────────
    for npz_path in sorted(npz_files):
        m = re.search(r'ctx(\d+)', npz_path.stem)
        if not m:
            if progress_bar: progress_bar.update(1)
            continue
        ctx_id = int(m.group(1))

        cache_path = cache_dir / f'ctx{ctx_id:03d}.npz'
        if cache_path.exists() and not overwrite:
            # Contexto já processado — só lê ch_order se ainda não temos
            if ch_order is None:
                try:
                    tmp = np.load(cache_path, allow_pickle=True)
                    ch_order = list(tmp['ch_order'])
                except Exception:
                    pass
            if progress_bar:
                progress_bar.set_postfix_str(f'ctx{ctx_id:03d} cached')
                progress_bar.update(1)
            continue

        # ── Lê o .npz do NB1 ────────────────────────────────────────────────
        try:
            data = np.load(npz_path, allow_pickle=True)
        except Exception as e:
            tqdm.write(f'    Erro {npz_path.name}: {e}')
            if progress_bar: progress_bar.update(1)
            continue

        ch_orig   = list(data['ch_names'])
        sfreq     = float(data['sfreq'])
        inter_raw = data['inter'].astype(np.float32)
        pre_raw   = data['pre'].astype(np.float32)

        # ── Mapeamento de canais ─────────────────────────────────────────────
        if is_seizeit:
            n_use   = min(2, inter_raw.shape[0])
            inter_m = inter_raw[:n_use]
            pre_m   = pre_raw[:n_use]
            if ch_order is None:
                ch_order = ch_orig[:n_use]
        else:
            names_o, idx_o, n_miss = map_channels(ch_orig, priority)
            n_miss_total += n_miss
            n_out   = len(names_o)
            inter_m = np.zeros((n_out, inter_raw.shape[1]), dtype=np.float32)
            pre_m   = np.zeros((n_out, pre_raw.shape[1]),   dtype=np.float32)
            for j, idx in enumerate(idx_o):
                if idx is not None and idx < inter_raw.shape[0]:
                    inter_m[j] = inter_raw[idx]
                    pre_m[j]   = pre_raw[idx]
            if ch_order is None:
                ch_order = names_o

        # ── Segmentação + features (40min completo) ──────────────────────────
        if inter_m.shape[1] < int(WIN_SEG_S * sfreq):
            tqdm.write(f'    ctx{ctx_id:03d}: sinal curto demais, ignorado')
            if progress_bar: progress_bar.update(1)
            continue

        Xi, ti = segment_signal(inter_m, sfreq)
        Xp, tp = segment_signal(pre_m,   sfreq)

        if len(Xi) == 0 or len(Xp) == 0:
            if progress_bar: progress_bar.update(1)
            continue

        pre_dur_s     = pre_m.shape[1] / sfreq
        tp_from_onset = tp - pre_dur_s          # negativo: segundos antes do onset

        inter_dur_s   = inter_m.shape[1] / sfreq
        ti_from_end   = ti - inter_dur_s         # negativo: segundos antes do fim do interictal
        # Mesmo referencial que t_pre: ti_from_end=-W_s seleciona os últimos W_s
        # segundos do interictal, independentemente da duração de cada contexto.
        # Isso corrige o bug de usar t_inter.max() global no NB4, que cortava
        # errado quando contextos tinham durações de interictal diferentes.

        # ── Salva checkpoint deste contexto ──────────────────────────────────
        np.savez_compressed(
            cache_path,
            Xi       = Xi,
            Xp       = Xp,
            ti       = ti_from_end,
            tp       = tp_from_onset,
            ctx_id   = np.array(ctx_id),
            ch_order = np.array(ch_order),
        )

        if progress_bar:
            progress_bar.set_postfix_str(f'ctx{ctx_id:03d} ✓')
            progress_bar.update(1)

    # ── Merge de todos os contextos (cache → arquivo final) ─────────────────
    cache_files = sorted(cache_dir.glob('ctx*.npz'))
    if not cache_files:
        return None, 'sem contextos processáveis'

    all_Xi, all_Xp          = [], []
    all_ti, all_tp          = [], []
    all_cti, all_ctp        = [], []

    for cf in cache_files:
        try:
            # Usa context manager para garantir que o arquivo seja fechado
            # antes de tentar deletá-lo (necessário no Windows)
            with np.load(cf, allow_pickle=True) as c:
                ctx_id = int(c['ctx_id'])
                xi = c['Xi'].copy(); ti = c['ti'].copy()
                xp = c['Xp'].copy(); tp = c['tp'].copy()
                ch_o = list(c['ch_order'])
            # Arquivo fechado — agora é seguro acumular e depois deletar
            all_Xi.append(xi); all_ti.append(ti)
            all_Xp.append(xp); all_tp.append(tp)
            all_cti.append(np.full(len(xi), ctx_id, dtype=np.int32))
            all_ctp.append(np.full(len(xp), ctx_id, dtype=np.int32))
            if ch_order is None:
                ch_order = ch_o
        except Exception as e:
            tqdm.write(f'    Erro merge {cf.name}: {e}')

    if not all_Xi:
        return None, 'merge vazio'

    X_inter       = np.vstack(all_Xi)
    X_pre         = np.vstack(all_Xp)
    t_inter       = np.concatenate(all_ti)
    t_pre         = np.concatenate(all_tp)
    ctx_ids_inter = np.concatenate(all_cti)
    ctx_ids_pre   = np.concatenate(all_ctp)

    n_ch  = X_inter.shape[1] // N_FEATS_CH
    meta  = json.dumps({
        'dataset':       dataset,
        'paciente':      pat,
        'is_seizeit2':   is_seizeit,
        'sfreq':         SFREQ,
        'win_seg_s':     WIN_SEG_S,
        'step_seg_s':    STEP_SEG_S,
        'n_feats_ch':    N_FEATS_CH,
        'n_ch':          n_ch,
        'ch_order':      list(ch_order) if ch_order else [],
        'n_missing_ch':  n_miss_total,
        'n_contextos':   len(set(ctx_ids_pre)),
        'n_win_pre':     int(len(X_pre)),
        'n_win_inter':   int(len(X_inter)),
        't_pre_range_s':  [float(t_pre.min()),   float(t_pre.max())],
        't_inter_range_s': [float(t_inter.min()), float(t_inter.max())],
        'level_slices':  (
            {k: v for k, v in LEVEL_SLICES.items() if v <= X_inter.shape[1]}
            if not is_seizeit else {'R0': int(X_inter.shape[1])}
        ),
    })

    np.savez_compressed(
        out_path,
        X_inter       = X_inter,
        X_pre         = X_pre,
        t_inter       = t_inter,
        t_pre         = t_pre,
        ctx_ids_inter = ctx_ids_inter,
        ctx_ids_pre   = ctx_ids_pre,
        meta          = np.array(meta),
    )

    # Limpa cache após merge bem-sucedido
    for cf in cache_files:
        try:
            cf.unlink(missing_ok=True)
        except PermissionError:
            pass  # Windows: arquivo ainda em uso — ignora, não é crítico
    try:
        cache_dir.rmdir()
    except OSError:
        pass

    msg = (f'OK — {len(set(ctx_ids_pre))} ctx | '
           f'X_pre={X_pre.shape} X_inter={X_inter.shape} | '
           f't_pre=[{t_pre.min()/60:.1f},{t_pre.max()/60:.1f}]min '
           f't_inter=[{t_inter.min()/60:.1f},{t_inter.max()/60:.1f}]min')
    return str(out_path), msg

## 5. Execução — todos os pacientes

In [6]:
# Agrupa .npz por (dataset, paciente)
npz_by_pat = defaultdict(list)
for p in sorted(SIGNAL_DIR.glob('*.npz')):
    parts = p.stem.split('__')
    if len(parts) >= 2:
        npz_by_pat[(parts[0], parts[1])].append(p)

total_pats = len(npz_by_pat)
total_ctx  = sum(len(v) for v in npz_by_pat.values())
print(f'Pacientes : {total_pats}')
print(f'Contextos : {total_ctx}')
print()
print('Checkpoint por contexto ativo — se interrompido, reinicie a célula.')
print('Contextos já processados ficam em data/features/.cache/ e são pulados.\n')

OVERWRITE = False   # True para forçar regeneração completa

n_ok = n_skip = n_err = 0

by_dataset = defaultdict(list)
for (ds, pat), files in sorted(npz_by_pat.items()):
    by_dataset[ds].append((pat, files))

# ── Barra 1: datasets ──────────────────────────────────────────────────────
pbar_ds = tqdm(sorted(by_dataset.keys()),
               desc='Datasets', unit='ds', position=0, leave=True,
               colour='blue', ncols=80)

for dataset in pbar_ds:
    pats = by_dataset[dataset]
    pbar_ds.set_postfix_str(dataset)

    # ── Barra 2: pacientes ────────────────────────────────────────────────
    pbar_pat = tqdm(pats, desc=f'  {dataset:<10}', unit='pac',
                    position=1, leave=False, colour='green', ncols=80)

    for pat, files in pbar_pat:
        n_cached = sum(
            1 for f in files
            if (m := re.search(r'ctx(\d+)', f.stem))
            if (FEAT_DIR / '.cache' / f'{dataset}__{pat}' /
                f'ctx{int(m.group(1)):03d}.npz').exists()
        )
        n_pending = len(files) - n_cached
        pbar_pat.set_postfix_str(
            f'{pat}  {n_pending}/{len(files)} ctx pendentes'
        )

        # ── Barra 3: contextos dentro do paciente ─────────────────────────
        pbar_ctx = tqdm(
            total=len(files),
            initial=n_cached,
            desc=f'    {pat:<12}', unit='ctx',
            position=2, leave=False, colour='yellow', ncols=80,
        )

        result, msg = process_patient(dataset, pat, files,
                                      overwrite=OVERWRITE,
                                      progress_bar=pbar_ctx)
        pbar_ctx.close()

        if result:
            tag = 'SKIP' if 'ja existe' in msg else ' OK '
            if 'ja existe' in msg: n_skip += 1
            else: n_ok += 1
        else:
            tag = 'ERR '; n_err += 1

        tqdm.write(f'  [{tag}] {dataset:<10} {pat:<12}  {msg}')

    pbar_pat.close()

pbar_ds.close()

print()
print(f'Gerados: {n_ok}  |  Já existiam: {n_skip}  |  Erros: {n_err}')
print(f'Arquivos em {FEAT_DIR}: {len(list(FEAT_DIR.glob("*.npz")))}')

Pacientes : 24
Contextos : 104

Checkpoint por contexto ativo — se interrompido, reinicie a célula.
Contextos já processados ficam em data/features/.cache/ e são pulados.



Datasets:   0%|                                   | 0/4 [00:00<?, ?ds/s, CHBMIT]





                                                                                
Datasets:   0%|                                   | 0/4 [00:32<?, ?ds/s, CHBMIT]

  [ OK ] CHBMIT     chb02         OK — 2 ctx | X_pre=(318, 323) X_inter=(478, 323) | t_pre=[-39.8,-0.3]min t_inter=[-59.8,-0.2]min








                                                                                
Datasets:   0%|                                   | 0/4 [02:11<?, ?ds/s, CHBMIT]

  [ OK ] CHBMIT     chb04         OK — 2 ctx | X_pre=(318, 323) X_inter=(1917, 323) | t_pre=[-39.8,-0.3]min t_inter=[-239.8,-0.2]min


...

  [ OK ] Siena      PN13          OK — 3 ctx | X_pre=(477, 323) X_inter=(613, 323) | t_pre=[-39.8,-0.2]min t_inter=[-55.6,-0.3]min










                                                                                
Datasets:  75%|████████████████████▎      | 3/4 [22:57<05:19, 319.16s/ds, Siena]

  [ OK ] Siena      PN14          OK — 3 ctx | X_pre=(477, 323) X_inter=(899, 323) | t_pre=[-39.8,-0.2]min t_inter=[-119.8,-0.2]min








                                                                                
Datasets: 100%|███████████████████████████| 4/4 [23:23<00:00, 350.91s/ds, Siena]

  [ OK ] Siena      PN16          OK — 2 ctx | X_pre=(318, 323) X_inter=(487, 323) | t_pre=[-39.8,-0.2]min t_inter=[-72.7,-0.4]min

Gerados: 24  |  Já existiam: 0  |  Erros: 0
Arquivos em data\features: 24


## 6. Validação — tabela de resumo

In [7]:
print('VALIDAÇÃO DOS ARQUIVOS DE FEATURES')
print('='*80)
print(f'  {"Dataset":<10} {"Paciente":<12} {"Ctx":<5} '
      f'{"X_pre":<18} {"X_inter":<18} {"t_pre (min)":<18} {"Nível máx"}')
print('  ' + '-'*78)

for fp in sorted(FEAT_DIR.glob('*.npz')):
    try:
        d    = np.load(fp, allow_pickle=True)
        meta = json.loads(str(d['meta']))
        tp   = d['t_pre']
        ti   = d['t_inter']
        max_lv = max(meta['level_slices'],
                     key=lambda k: meta['level_slices'][k])
        n_ctx = len(set(d['ctx_ids_pre']))
        print(f'  {meta["dataset"]:<10} {meta["paciente"]:<12} {n_ctx:<5} '
              f'{str(d["X_pre"].shape):<18} {str(d["X_inter"].shape):<18} '
              f't_pre=[{tp.min()/60:.1f},{tp.max()/60:.1f}]  t_inter=[{ti.min()/60:.1f},{ti.max()/60:.1f}]  {max_lv}')
    except Exception as e:
        print(f'  ERRO {fp.name}: {e}')

print('  ' + '-'*78)
print()
print('NOTA: t_pre são segundos antes do onset (negativo).')
print('      NB4 faz: X_pre[ t_pre >= -W_s ]  para janela W minutos.')

VALIDAÇÃO DOS ARQUIVOS DE FEATURES
  Dataset    Paciente     Ctx   X_pre              X_inter            t_pre (min)        Nível máx
  ------------------------------------------------------------------------------
  CHBMIT     chb02        2     (318, 323)         (478, 323)         t_pre=[-39.8,-0.3]  t_inter=[-59.8,-0.2]  R5
  CHBMIT     chb04        2     (318, 323)         (1917, 323)        t_pre=[-39.8,-0.3]  t_inter=[-239.8,-0.2]  R5
  CHBMIT     chb06        7     (1113, 323)        (6713, 323)        t_pre=[-39.8,-0.3]  t_inter=[-239.8,-0.2]  R5
  CHBMIT     chb07        3     (477, 323)         (2397, 323)        t_pre=[-39.8,-0.3]  t_inter=[-239.8,-0.2]  R5
  CHBMIT     chb08        4     (636, 323)         (956, 323)         t_pre=[-39.8,-0.3]  t_inter=[-59.8,-0.2]  R5
  CHBMIT     chb10        5     (795, 323)         (2396, 323)        t_pre=[-39.8,-0.3]  t_inter=[-120.1,-0.2]  R5
  CHBMIT     chb13        2     (318, 323)         (478, 323)         t_pre=[-39.8,-0.3]  t

## 7. Demonstração — como o NB4 deve usar este arquivo

In [8]:
# ── Janelas experimentais do NB4 (derivadas do NB2) ──────────────────────────
# Percentis P25, P50, P75, max da distribuição PRE_SEC (resultado do NB2):
WINDOWS_NB4 = {
    'W1':  8 * 60,   #  P25 ≈  7.7min → usa 8min
    'W2': 10 * 60,   #  P50 ≈  9.6min → usa 10min
    'W3': 13 * 60,   #  P75 ≈ 13.3min → usa 13min
    'W4': 19 * 60,   #  max ≈ 18.7min → usa 19min
    # 'W5': pre_sec_do_paciente  ← lido do preictal_estimate.json pelo NB4
}

# ── Exemplo de uso (com arquivo existente) ────────────────────────────────────
example = next((f for f in sorted(FEAT_DIR.glob('*.npz'))
                if 'SeizeIT2' not in f.name), None)

if example:
    d    = np.load(example, allow_pickle=True)
    meta = json.loads(str(d['meta']))
    X_pre         = d['X_pre']
    X_inter       = d['X_inter']
    t_pre         = d['t_pre']          # segundos antes do onset (< 0)
    t_inter       = d['t_inter']        # segundos desde início do segmento inter (> 0)
    ctx_ids_pre   = d['ctx_ids_pre']
    ctx_ids_inter = d['ctx_ids_inter']

    # t_inter é negativo (segundos antes do fim do interictal), igual ao t_pre
    # → mesmo filtro para ambos: >= -W_s

    print(f'Arquivo: {example.name}')
    print(f'Dataset/paciente: {meta["dataset"]}/{meta["paciente"]}')
    print(f'X_pre completo  : {X_pre.shape}   (t_pre de {t_pre.min()/60:.1f}min a {t_pre.max()/60:.1f}min)')
    print(f'X_inter completo: {X_inter.shape}  (t_inter de {t_inter.min()/60:.1f}min a {t_inter.max()/60:.1f}min)')
    print()

    for wlabel, W_s in WINDOWS_NB4.items():
        # Mesmo filtro para pré-ictal e interictal: últimos W_s segundos
        # t_pre  >= -W_s  →  janela pré-ictal
        # t_inter >= -W_s  →  janela interictal equivalente
        mask_pre   = t_pre   >= -W_s
        Xp_W       = X_pre[mask_pre]
        ctx_pre_W  = ctx_ids_pre[mask_pre]

        mask_inter = t_inter >= -W_s
        Xi_W       = X_inter[mask_inter]
        ctx_int_W  = ctx_ids_inter[mask_inter]

        print(f'  {wlabel} (W={W_s//60}min):  '
              f'X_pre={Xp_W.shape}  X_inter={Xi_W.shape}  '
              f'Δ={abs(len(Xp_W)-len(Xi_W))} jan')

    print()
    print('Fatiamento de canais por nível:')
    for lvl, n_cols in sorted(meta["level_slices"].items(),
                               key=lambda x: x[1], reverse=True):
        n_ch = n_cols // N_FEATS_CH
        print(f'  {lvl}: X_pre[:, :{n_cols}]  → {n_ch} canais × 19 features')
else:
    print('Execute a célula 5 primeiro para gerar os arquivos de features.')

Arquivo: CHBMIT__chb02.npz
Dataset/paciente: CHBMIT/chb02
X_pre completo  : (318, 323)   (t_pre de -39.8min a -0.3min)
X_inter completo: (478, 323)  (t_inter de -59.8min a -0.2min)

  W1 (W=8min):  X_pre=(62, 323)  X_inter=(64, 323)  Δ=2 jan
  W2 (W=10min):  X_pre=(78, 323)  X_inter=(80, 323)  Δ=2 jan
  W3 (W=13min):  X_pre=(102, 323)  X_inter=(104, 323)  Δ=2 jan
  W4 (W=19min):  X_pre=(150, 323)  X_inter=(152, 323)  Δ=2 jan

Fatiamento de canais por nível:
  R5: X_pre[:, :323]  → 17 canais × 19 features
  R4: X_pre[:, :285]  → 15 canais × 19 features
  R3: X_pre[:, :247]  → 13 canais × 19 features
  R2: X_pre[:, :209]  → 11 canais × 19 features
  R1: X_pre[:, :76]  → 4 canais × 19 features
  R0: X_pre[:, :38]  → 2 canais × 19 features


## 8. Exportação do catálogo de features para o NB4

In [9]:
# features_map.json — catálogo de todos os pacientes processados
# O NB4 usa este arquivo para descobrir quais .npz existem e seus metadados

features_map = []
for fp in sorted(FEAT_DIR.glob('*.npz')):
    try:
        d    = np.load(fp, allow_pickle=True)
        meta = json.loads(str(d['meta']))
        features_map.append({
            'dataset':       meta['dataset'],
            'paciente':      meta['paciente'],
            'path':          str(fp),
            'n_ctx':         meta['n_contextos'],
            'n_win_pre':     meta['n_win_pre'],
            'n_win_inter':   meta['n_win_inter'],
            'shape_pre':     list(d['X_pre'].shape),
            'shape_inter':   list(d['X_inter'].shape),
            't_pre_range_min': [round(float(d['t_pre'].min())/60,1),
                                round(float(d['t_pre'].max())/60,1)],
            'level_slices':  meta['level_slices'],
            'ch_order':      meta['ch_order'],
            'is_seizeit2':   meta['is_seizeit2'],
            'n_missing_ch':  meta['n_missing_ch'],
        })
    except Exception as e:
        print(f'Erro {fp.name}: {e}')

# Inclui as janelas do NB4 (derivadas do NB2) como referência
nb4_windows = {
    'W1':  8 * 60, 'W2': 10 * 60, 'W3': 13 * 60, 'W4': 19 * 60,
    'W5':  'PRE_SEC_individual (preictal_estimate.json)',
}

output = {'nb4_windows_s': nb4_windows, 'patients': features_map}
out_path = ROOT / 'features_map.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'Exportados {len(features_map)} pacientes para: {out_path}')
from collections import Counter
for ds, n in Counter(e['dataset'] for e in features_map).items():
    print(f'  {ds}: {n} pacientes')

try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    print(f'Arquivo: {out_path.resolve()}')

Exportados 24 pacientes para: data\features_map.json
  CHBMIT: 7 pacientes
  Mendeley: 3 pacientes
  SeizeIT2: 7 pacientes
  Siena: 7 pacientes
Arquivo: D:\TCC\data\features_map.json
